# 문제 2-2 : 영화 리뷰 감정 분석기

영화 리뷰를 "긍정", "부정", "보통" 중 하나로 분류합니다.

- `EnumOutputParser` 와 `Enum` 클래스 사용
- 여러 개의 테스트 리뷰로 검증
- (선택) 분석 실패 시 `OutputFixingParser` 로 자동 복구

### 공통 설정: API Key 로드 + LLM 생성 함수

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


In [2]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True, **kwargs) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
        **kwargs,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [3]:
from enum import Enum
from langchain_classic.output_parsers import EnumOutputParser  # langchain 1.x: classic 패키지에서 import
from langchain_core.prompts import ChatPromptTemplate


# 허용할 분류 값을 Enum 으로 정의 (값 = 모델이 답해야 하는 문자열)
class Sentiment(Enum):
    """영화 리뷰 감정 분류 값"""
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "보통"


# Enum 출력 파서 초기화
parser = EnumOutputParser(enum=Sentiment)

# 출력 형식 지침 확인: "Select one of the following options: 긍정, 부정, 보통"
format_instructions = parser.get_format_instructions()
print(format_instructions)

Select one of the following options: 긍정, 부정, 보통


In [4]:
# 프롬프트 템플릿 설정 (형식 지침을 미리 주입)
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 영화 리뷰 감정 분석 전문가입니다. 리뷰를 읽고 감정을 분류하세요. "
               "칭찬이 중심이면 긍정, 불만이 중심이면 부정, 좋지도 나쁘지도 않으면 보통입니다. "
               "설명 없이 값만 답하세요.\n{format_instructions}"),
    ("user", "리뷰: {review}"),
]).partial(format_instructions=format_instructions)

# 분류 작업이므로 temperature 를 0 으로 설정
llm = get_llm(temperature=0)

# 체인 생성: 프롬프트 → 모델 → Enum 파서
chain = prompt | llm | parser

[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [5]:
from langchain_core.exceptions import OutputParserException

# 감정 이모지 (결과를 깔끔하게 출력하기 위한 용도)
EMOJI = {Sentiment.POSITIVE: "😊", Sentiment.NEGATIVE: "😞", Sentiment.NEUTRAL: "😐"}

reviews = [
    "이 영화 정말 재미없어요. 시간 낭비였습니다.",
    "배우들의 연기가 훌륭하고 스토리도 감동적이었어요!",
    "그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.",
    "영상미는 압도적인데 결말이 너무 허무해서 아쉬웠어요.",
    "올해 본 영화 중 최고! 두 번 더 보러 갈 거예요.",
]

print(f"{'번호':<4}{'결과':<8}리뷰")
print("=" * 70)
for i, review in enumerate(reviews, 1):
    try:
        result = chain.invoke({"review": review})  # 결과는 Sentiment Enum 멤버
        print(f"{i:<5}{EMOJI[result]} {result.value:<5}{review}")
    except OutputParserException as e:
        # 모델이 Enum 에 없는 값으로 답하면 파싱 오류 발생
        print(f"{i:<5}❌ 분석 실패: {str(e).splitlines()[0]}")

번호  결과      리뷰
1    😞 부정   이 영화 정말 재미없어요. 시간 낭비였습니다.
2    😊 긍정   배우들의 연기가 훌륭하고 스토리도 감동적이었어요!
3    😐 보통   그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.
4    😞 부정   영상미는 압도적인데 결말이 너무 허무해서 아쉬웠어요.
5    😊 긍정   올해 본 영화 중 최고! 두 번 더 보러 갈 거예요.


### (선택) OutputFixingParser 로 분석 실패 자동 복구

In [6]:
from langchain_classic.output_parsers import OutputFixingParser

# 기존 Enum 파서를 감싸서 파싱 실패 시 LLM 으로 응답을 자동 수정
fixing_parser = OutputFixingParser.from_llm(llm=llm, parser=parser, max_retries=1)

# 모델이 형식을 어기고 영어로 답한 경우를 가정
bad_completion = "Positive"

try:
    parser.parse(bad_completion)
except OutputParserException as e:
    print("1) 기본 Enum 파서 오류:", str(e).splitlines()[0])

fixed = fixing_parser.parse(bad_completion)
print(f"2) OutputFixingParser 수정 결과: {fixed} / 값: {fixed.value}")

# 체인에 적용: 프롬프트 → 모델 → 오류 자동 복구 파서
fixing_chain = prompt | llm | fixing_parser
print("3) fixing_chain 결과:", fixing_chain.invoke({"review": reviews[1]}).value)

1) 기본 Enum 파서 오류: Response 'Positive' is not one of the expected values: ['긍정', '부정', '보통']
2) OutputFixingParser 수정 결과: Sentiment.NEUTRAL / 값: 보통
3) fixing_chain 결과: 긍정
